### 🧠 Chat with My PDFs — Local Inference

### Install dependencies


In [1]:
# !pip install -U numpy==1.26.4 scikit-learn==1.5.1 sentence-transformers==3.0.1 faiss-cpu==1.8.0.post1
# !pip install -U pillow==9.5.0 torchvision==0.15.2 torch==2.0.1

### Import Libraries

In [1]:
import os
import fitz  # PyMuPDF
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import streamlit as st

c:\Users\sahup\AppData\Local\Programs\Python\Python310\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
Disabling PyTorch because PyTorch >= 2.1 is required but found 2.0.1


AttributeError: module 'torch' has no attribute 'compiler'

### STEP 1: Load PDF and Extract Text

In [ ]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text")
    return text

pdf_path = "sample.pdf"  # 📝 Replace with your PDF path
text = extract_text_from_pdf(pdf_path)
print("✅ Extracted text length:", len(text))


### STEP 2: Chunk the Text

In [ ]:
def chunk_text(text, chunk_size=500):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

chunks = chunk_text(text)
print(f"✅ Created {len(chunks)} chunks.")

### STEP 3: Create Embeddings

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

### STEP 4: Create and Save FAISS Index

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

### Save both index and chunks

In [ ]:
faiss.write_index(index, "pdf_index.faiss")
with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("✅ Embeddings stored in FAISS index")


### STEP 5: Search Function

In [ ]:
def search_similar_chunks(query, model, index, chunks, top_k=3):
    query_emb = model.encode([query]).astype("float32")
    distances, indices = index.search(query_emb, top_k)
    results = [chunks[i] for i in indices[0]]
    return results

### STEP 6: Ask Questions!

In [ ]:
query = "Where is the store located?"
results = search_similar_chunks(query, model, index, chunks)
for i, res in enumerate(results):
    print(f"\nResult {i+1}:\n{res[:500]}...")
